In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Define the model path and settings
model_name = "unsloth/gemma-4-E2B"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
    device_map = "auto",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj","v_proj",
                "k_proj","o_proj",
                "gate_proj","up_proj",
                "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/abhijit_instruct.jsonl", split="train")

abhijit_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are Abhijit Chavda. Answer the user's question with historical depth and an assertive tone.<|eot_id|><|start_header_id|>user<|end_header_id|>
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{}<|eot_id|>"""

def formatting_prompts_func(examples):
    inputs  = examples["input"]
    outputs = examples["output"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        text = abhijit_prompt.format(input_text, output_text)
        texts.append(text)
    return { "text" : texts, }
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Final VRAM-optimized settings
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Dropped to 1 to prevent OOM
        gradient_accumulation_steps = 8, # Doubled to keep effective batch size at 8
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",          # Keep 8-bit for memory efficiency
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        # Force the model to be even more stingy with memory
        gradient_checkpointing = True,
        save_total_limit = 1,          # Don't waste disk space on old checkpoints
    ),
)

# Start training again
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5201 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,201 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 62,078,976 of 5,185,256,992 (1.20% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,12.422066
2,12.715078
3,12.853910
4,12.721223
5,12.408231
6,12.916449
7,12.994769
8,11.293483
9,9.824120
10,8.327352


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
# Use .save_pretrained() for the LoRA adapters
model.save_pretrained("abhijit_chavda_lora")
tokenizer.save_pretrained("abhijit_chavda_lora")

# Zip the folder for download
import os
os.system("zip -r abhijit_chavda_lora.zip abhijit_chavda_lora")

# Download to your computer
from google.colab import files
files.download("abhijit_chavda_lora.zip")

Unsloth: Restored added_tokens_decoder metadata in abhijit_chavda_lora/tokenizer_config.json.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Unsloth supports .save_pretrained_gguf() for CPU-optimized formats
model.save_pretrained_gguf(
    "abhijit_chavda_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

# Zip and download the GGUF version
os.system("zip -r abhijit_chavda_gguf.zip abhijit_chavda_gguf")
files.download("abhijit_chavda_gguf.zip")

Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in abhijit_chavda_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

In [ ]:
# 1. Re-install the essentials
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes



In [ ]:
from unsloth import FastLanguageModel
import torch
# 2. Load the base model and your local folder
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    max_seq_length = 512,
    load_in_4bit = True,
)

# This 'point' the model to your surviving folder
model = FastLanguageModel.get_peft_model(model, model_name = "/content/abhijit_chavda_lora")
FastLanguageModel.for_inference(model)

# 3. Chat Function
kenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|start_of_turn|>user\n' + message['content'] + '<|end_of_turn|>\n' }}{% elif message['role'] == 'system' %}{{ '<|start_of_turn|>system\n' + message['content'] + '<|end_of_turn|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|start_of_turn|>model\n' + message['content'] + '<|end_of_turn|>\n' }}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|start_of_turn|>model\n' }}{% endif %}{% endfor %}"""

def ask_abhijit(question):
    # Gemma 4 E2B expects this multimodal-friendly list format
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are Abhijit Chavda. Answer with historical depth and assertiveness."}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": question}]
        }
    ]

    # Apply the template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    print(f"\n--- Abhijit AI Response ---\n")
    _ = model.generate(
        input_ids = inputs,
        streamer = text_streamer,
        max_new_tokens = 512,
        temperature = 0.5,
        use_cache = True
    )

# Now try the question again
ask_abhijit("What do you think about the geopolitical future of Pakistan?")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
import torch
import gc

# Clear Python's garbage collector
gc.collect()

# Clear PyTorch's CUDA cache
torch.cuda.empty_cache()

# Optional: Check if memory is actually free
!nvidia-smi

Sun Apr 26 09:51:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |    8911MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Force kill the process holding the GPU
!kill -9 19859

# Wait 2 seconds, then check memory again
import time
time.sleep(2)
!nvidia-smi